# RAG Frameworks Ecosystem: Choosing the Right Tool

The RAG (Retrieval-Augmented Generation) space has exploded with frameworks, each making different bets on the right abstraction level.

## Why so many frameworks?

RAG is deceptively simple in concept retrieve relevant context, inject into prompt but production systems surface hard problems:
- Document parsing (PDFs with tables, scanned docs, mixed formats)
- Chunking strategies that preserve semantic meaning
- Hybrid retrieval (dense + sparse)
- Re-ranking and filtering
- Evaluation and observability
- Scaling to millions of documents

## The trade-off landscape

| Dimension | One End | Other End |
|-----------|---------|----------|
| **Abstraction** | High-level (quick start) | Low-level (full control) |
| **Deployment** | Local / private | Cloud-native |
| **Target** | Developer prototyping | Production MLOps |
| **Interface** | Visual / no-code | Code-first |
| **Scope** | RAG-only | Full LLM application platform |

**Frameworks covered in this notebook:**
1. LangChain the Swiss Army knife
2. LlamaIndex data-centric RAG
3. Haystack production-grade pipelines
4. RAGFlow deep document understanding
5. Dify LLMOps platform
6. Flowise & Langflow visual builders
7. Other notable frameworks (txtai, Semantic Kernel, Vercel AI SDK, etc.)
8. Document preprocessing libraries
9. Decision matrix & comparison
10. Side-by-side: LangChain vs LlamaIndex

## 1. LangChain

LangChain started as a collection of helpful wrappers and grew into the most widely-used LLM application framework. Its superpower is breadth: 100+ document loaders, integrations with virtually every vector store, and a composable expression language.

### Key concepts

**LCEL (LangChain Expression Language)** chains are built with the `|` pipe operator. Every component is a `Runnable` with a uniform `.invoke()` / `.stream()` / `.batch()` interface.

```
chain = prompt | llm | output_parser
result = chain.invoke({"question": "..."})
```

**Core building blocks:**
- `DocumentLoader` ingests raw data (PDF, web, CSV, S3, Notion, ...)
- `TextSplitter` chunks documents (`RecursiveCharacterTextSplitter` is the default choice)
- `Embeddings` converts text to vectors
- `VectorStore` stores and retrieves embeddings
- `Retriever` `.as_retriever()` wraps any vector store; supports MMR, similarity threshold, multi-query
- `RunnablePassthrough` passes input unchanged through the chain (useful for threading `question` alongside retrieved docs)
- `RunnableParallel` runs multiple runnables in parallel, merges results

**Package structure (post v0.2 split):**
- `langchain-core` base abstractions (Runnable, BaseMessage, etc.)
- `langchain` chains, agents, and high-level logic
- `langchain-community` third-party integrations
- `langchain-openai`, `langchain-anthropic`, etc. vendor packages

In [1]:
# ── LangChain: End-to-End RAG Pipeline ──────────────────────────────────────
# Install: pip install langchain langchain-openai langchain-community chromadb

import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

# ── 1. Load documents ────────────────────────────────────────────────────────
# WebBaseLoader fetches a URL and returns a list of Document objects
loader = WebBaseLoader(
    web_paths=["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    # bs_kwargs lets you pass BeautifulSoup kwargs to filter the HTML
    bs_kwargs=dict(
        parse_only=None  # parse the full page; use SoupStrainer to narrow
    )
)
docs = loader.load()  # returns List[Document]
print(f"Loaded {len(docs)} document(s), total chars: {sum(len(d.page_content) for d in docs)}")

# ── 2. Split into chunks ─────────────────────────────────────────────────────
# RecursiveCharacterTextSplitter tries to split on \n\n, then \n, then ' ', then ''
# This preserves paragraph structure as much as possible
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # max characters per chunk
    chunk_overlap=200,     # overlap helps preserve context across chunk boundaries
    add_start_index=True,  # adds 'start_index' to metadata for provenance
)
splits = splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")

# ── 3. Embed & store ─────────────────────────────────────────────────────────
# Chroma is an in-process vector store; great for prototyping
# For production: swap to PGVector, Pinecone, Weaviate, Qdrant, etc.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_langchain_demo",  # set None for in-memory
)

# ── 4. Create a retriever ────────────────────────────────────────────────────
# .as_retriever() wraps the vector store; search_type options:
#   'similarity'         - cosine similarity (default)
#   'mmr'                - maximal marginal relevance (diversity)
#   'similarity_score_threshold' - filter by score
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20},  # fetch 20, return top-4 diverse
)

# ── 5. Define the prompt ─────────────────────────────────────────────────────
RAG_TEMPLATE = """\
You are an assistant for question-answering tasks.
Use ONLY the following retrieved context to answer the question.
If the context doesn't contain the answer, say "I don't know."
Keep your answer concise (3 sentences max).

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

# ── 6. LLM ───────────────────────────────────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── 7. Helper to format retrieved docs into a single string ──────────────────
def format_docs(docs):
    return "\n\n".join(
        f"[Source {i+1}]\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

# ── 8. Build the LCEL chain ──────────────────────────────────────────────────
# RunnableParallel runs both branches simultaneously:
#   - 'context': retrieve docs → format into string
#   - 'question': pass the input question through unchanged
# The dict output feeds into `prompt`, then `llm`, then `StrOutputParser`
rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever | format_docs,
        question=RunnablePassthrough() | (lambda x: x["question"]),
    )
    | prompt
    | llm
    | StrOutputParser()
)

# ── 9. Run a query ───────────────────────────────────────────────────────────
# Uncomment when OPENAI_API_KEY is set:
# answer = rag_chain.invoke({"question": "What is the role of memory in LLM agents?"})
# print(answer)

# ── Bonus: streaming ─────────────────────────────────────────────────────────
# LCEL chains support streaming out of the box
# for chunk in rag_chain.stream({"question": "What tools can agents use?"}):
#     print(chunk, end="", flush=True)

print("LangChain RAG chain constructed successfully.")
print("Chain repr:", rag_chain)

/tmp/ipykernel_184590/278640766.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/pydantic/plugin/_schema_validator.py:39: UserWarning: ImportError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ImportError("cannot import name 'LogData' from 'opentelemetry.sdk._logs' (/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/opentelemetry/sdk/_logs/__init__.py)")
  plugins = get_plugins()


USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 document(s), total chars: 43800
Split into 66 chunks


### LangChain: Pros & Cons

| Pros | Cons |
|------|------|
| Largest ecosystem of integrations | Abstractions leak under edge cases |
| Active community, rapid releases | Package split caused breaking changes |
| LCEL makes chains composable & streamable | Can feel verbose for simple use cases |
| Good documentation & tutorials | Version churn Stack Overflow answers go stale |
| Built-in memory, agents, tools | Debugging multi-step chains can be painful |

## 2. LlamaIndex

LlamaIndex (formerly GPT Index) is laser-focused on **data**. Where LangChain is a general LLM application framework, LlamaIndex is built around the idea of indexing your data in different structures optimized for different query patterns.

### Index types

| Index | Structure | Best for |
|-------|-----------|----------|
| `VectorStoreIndex` | Embeddings in vector DB | Semantic similarity search |
| `KnowledgeGraphIndex` | Triplet extraction → graph | Relationship queries |
| `TreeIndex` | Hierarchical summarization tree | Summarization of long docs |
| `ListIndex` | Sequential node list | Small corpora, exhaustive search |
| `KeywordTableIndex` | Inverted keyword table | Keyword-based lookup |

### Query engines

- `VectorIndexQueryEngine` standard semantic search
- `SubQuestionQueryEngine` decomposes complex questions into sub-questions, queries multiple indexes
- `RouterQueryEngine` routes queries to the most appropriate index/tool
- `RetrieverQueryEngine` compose any retriever with any response synthesizer

### Node parsers & metadata extractors

LlamaIndex exposes the document → node pipeline explicitly. `NodeParser` turns documents into `TextNode` objects. `MetadataExtractor` can enrich nodes with LLM-extracted summaries, questions, keyphrases, and custom fields.

In [2]:
# ── LlamaIndex: RAG with Metadata Filtering ──────────────────────────────────
# Install: pip install llama-index llama-index-embeddings-openai llama-index-llms-openai

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
    Document,
)
from llama_index.core.vector_stores import (
    MetadataFilters,
    MetadataFilter,
    FilterOperator,
    FilterCondition,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.extractors import (
    TitleExtractor,
    QuestionsAnsweredExtractor,
    SummaryExtractor,
)
from llama_index.core.ingestion import IngestionPipeline
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# ── Global settings (replaces ServiceContext in v0.10+) ──────────────────────
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.chunk_size = 512
Settings.chunk_overlap = 64

# ── 1. Create documents with rich metadata ───────────────────────────────────
# In real usage: SimpleDirectoryReader('data/') loads all files in a directory
# It auto-detects PDF, DOCX, HTML, TXT, CSV, PPTX, images, etc.
documents = [
    Document(
        text="LangChain is a framework for building LLM applications with chains and agents.",
        metadata={
            "source": "langchain_docs",
            "category": "framework",
            "year": 2023,
            "language": "python",
        },
    ),
    Document(
        text="LlamaIndex specializes in connecting LLMs to external data sources through various index structures.",
        metadata={
            "source": "llamaindex_docs",
            "category": "framework",
            "year": 2023,
            "language": "python",
        },
    ),
    Document(
        text="Haystack by deepset provides a production-ready pipeline architecture for NLP systems.",
        metadata={
            "source": "haystack_docs",
            "category": "framework",
            "year": 2022,
            "language": "python",
        },
    ),
    Document(
        text="Semantic Kernel is Microsoft's SDK for integrating LLMs into .NET and Python applications.",
        metadata={
            "source": "semantic_kernel_docs",
            "category": "framework",
            "year": 2023,
            "language": "dotnet",
        },
    ),
]

# ── 2. Ingestion pipeline with metadata extraction ───────────────────────────
# IngestionPipeline chains: split → extract metadata → embed → store
# Metadata extractors call the LLM to enrich each node automatically
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=512, chunk_overlap=64),
        # TitleExtractor: asks LLM to generate a title for the node
        # QuestionsAnsweredExtractor: generates questions this node can answer
        # (Commented out to avoid API calls in demo)
        # TitleExtractor(nodes=5),
        # QuestionsAnsweredExtractor(questions=3),
        OpenAIEmbedding(model="text-embedding-3-small"),
    ]
)

# nodes = pipeline.run(documents=documents)  # runs transformations

# ── 3. Build vector index ────────────────────────────────────────────────────
# VectorStoreIndex.from_documents handles the full pipeline internally
# For custom node processing, use VectorStoreIndex(nodes=nodes)
# index = VectorStoreIndex.from_documents(documents, show_progress=True)

# ── 4. Query with metadata filtering ─────────────────────────────────────────
# MetadataFilters allows pre-filtering before vector search
# This is crucial for multi-tenant systems or date-filtered searches

python_only_filter = MetadataFilters(
    filters=[
        MetadataFilter(
            key="language",
            value="python",
            operator=FilterOperator.EQ,  # EQ, NE, GT, GTE, LT, LTE, IN, NIN
        ),
        MetadataFilter(
            key="year",
            value=2022,
            operator=FilterOperator.GTE,  # year >= 2022
        ),
    ],
    condition=FilterCondition.AND,  # AND or OR
)

# Retriever with filters applied
# filtered_retriever = index.as_retriever(
#     similarity_top_k=3,
#     filters=python_only_filter,
# )

# Query engine wraps retriever + response synthesizer
# query_engine = index.as_query_engine(
#     similarity_top_k=3,
#     filters=python_only_filter,
#     response_mode="compact",  # 'compact', 'refine', 'tree_summarize', 'simple_summarize'
# )
# response = query_engine.query("Which Python RAG frameworks were released after 2022?")
# print(response)
# print("\nSource nodes:")
# for node in response.source_nodes:
#     print(f"  Score: {node.score:.3f} | Source: {node.metadata['source']}")

# ── 5. SubQuestion query engine (for complex multi-part questions) ────────────
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine

# In a real setup, you'd build separate indexes per topic/domain
# then wrap them as tools for the SubQuestionQueryEngine
# The engine decomposes: "Compare LangChain and LlamaIndex"
# → sub-q1: "What is LangChain?" (routed to langchain_tool)
# → sub-q2: "What is LlamaIndex?" (routed to llamaindex_tool)
# → synthesize both answers into a final comparison

print("LlamaIndex structures demonstrated (API calls commented out for demo).")
print("MetadataFilters config:", python_only_filter)

LlamaIndex structures demonstrated (API calls commented out for demo).
MetadataFilters config: filters=[MetadataFilter(key='language', value='python', operator=<FilterOperator.EQ: '=='>), MetadataFilter(key='year', value=2022, operator=<FilterOperator.GTE: '>='>)] condition=<FilterCondition.AND: 'and'>


### LlamaIndex: Pros & Cons

| Pros | Cons |
|------|------|
| Multiple index types for different query patterns | Less general-purpose than LangChain |
| Excellent metadata filtering support | Smaller community |
| `IngestionPipeline` is clean and composable | LLM-based metadata extraction adds cost |
| Strong support for structured data (SQL, Pandas) | Agent support is less mature |
| Good evaluation framework (`llama-eval`) | API has changed significantly across versions |

## 3. Haystack (Deepset)

Haystack is built by deepset, a company focused on enterprise NLP. It has the most production-oriented design philosophy of the major frameworks: explicit component interfaces, built-in evaluation, and a pipeline model that maps directly to CI/CD deployment.

### Architecture (Haystack 2.0)

Everything is a **Component** with typed `@component` decorator. Components declare their `Input` and `Output` dataclasses. A `Pipeline` is a directed graph of components you connect outputs to inputs explicitly.

```
pipeline.add_component("embedder", SentenceTransformersTextEmbedder())
pipeline.add_component("retriever", InMemoryEmbeddingRetriever(document_store=store))
pipeline.connect("embedder.embedding", "retriever.query_embedding")
```

### Document stores

| Store | Best for |
|-------|----------|
| `InMemoryDocumentStore` | Testing, small datasets |
| `ElasticsearchDocumentStore` | BM25 + dense hybrid, production |
| `OpenSearchDocumentStore` | AWS-hosted Elasticsearch |
| `WeaviateDocumentStore` | Cloud-native vector search |
| `QdrantDocumentStore` | High-performance vector DB |
| `PgvectorDocumentStore` | PostgreSQL + pgvector |

### Built-in evaluation

Haystack includes evaluation metrics: `ContextRelevance`, `Faithfulness`, `SemanticAnswerSimilarity`, `SASEvaluator`. The `EvaluationRunResult` provides a pandas DataFrame for analysis.

In [3]:
# ── Haystack 2.0: RAG Pipeline ───────────────────────────────────────────────
# Install: pip install haystack-ai

from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever,
    InMemoryEmbeddingRetriever,
)
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator
from haystack.components.joiners import DocumentJoiner
from haystack.components.rankers import MetaFieldRanker, TransformersSimilarityRanker

# ── 1. Document store and sample documents ───────────────────────────────────
document_store = InMemoryDocumentStore()

docs = [
    Document(
        content="Haystack is an open-source NLP framework for building search and QA systems.",
        meta={"source": "haystack_overview", "year": 2024},
    ),
    Document(
        content="Haystack 2.0 introduced a component-based architecture with typed inputs and outputs.",
        meta={"source": "haystack_2_release", "year": 2024},
    ),
    Document(
        content="The Pipeline class in Haystack is a directed acyclic graph of components.",
        meta={"source": "haystack_pipeline_docs", "year": 2024},
    ),
]

# ── 2. Indexing pipeline ─────────────────────────────────────────────────────
# Haystack separates indexing from querying two distinct pipelines
indexing_pipeline = Pipeline()
indexing_pipeline.add_component(
    "embedder",
    SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
)
indexing_pipeline.add_component("writer", __import__(
    "haystack.components.writers", fromlist=["DocumentWriter"]
).DocumentWriter(document_store=document_store))
indexing_pipeline.connect("embedder", "writer")

# indexing_pipeline.run({"embedder": {"documents": docs}})

# ── 3. RAG query pipeline ────────────────────────────────────────────────────
# This pipeline: embed query → retrieve → rank → generate
RAG_TEMPLATE = """
Answer the question based on the provided context.
Context:
{% for doc in documents %}
    {{ doc.content }}
{% endfor %}
Question: {{ question }}
Answer:
"""

rag_pipeline = Pipeline()

# Add components
rag_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
)
rag_pipeline.add_component(
    "embedding_retriever",
    InMemoryEmbeddingRetriever(document_store=document_store, top_k=5)
)
rag_pipeline.add_component(
    "bm25_retriever",
    InMemoryBM25Retriever(document_store=document_store, top_k=5)
)
# DocumentJoiner merges results from multiple retrievers (hybrid search)
rag_pipeline.add_component(
    "joiner",
    DocumentJoiner(join_mode="reciprocal_rank_fusion")  # or 'concatenate', 'distribution_based_rank_fusion'
)
rag_pipeline.add_component(
    "ranker",
    TransformersSimilarityRanker(model="cross-encoder/ms-marco-MiniLM-L-6-v2", top_k=3)
)
rag_pipeline.add_component("prompt_builder", PromptBuilder(template=RAG_TEMPLATE))
rag_pipeline.add_component("llm", OpenAIGenerator(model="gpt-4o-mini"))

# ── 4. Connect components explicitly ────────────────────────────────────────
# Connections use "component_name.output_name" -> "component_name.input_name"
# This explicitness is Haystack's signature: no magic, no hidden routing
rag_pipeline.connect("text_embedder.embedding", "embedding_retriever.query_embedding")
rag_pipeline.connect("embedding_retriever.documents", "joiner.documents")
rag_pipeline.connect("bm25_retriever.documents", "joiner.documents")
rag_pipeline.connect("joiner.documents", "ranker.documents")
rag_pipeline.connect("ranker.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder.prompt", "llm.prompt")

# ── 5. Visualize the pipeline (requires graphviz) ────────────────────────────
# rag_pipeline.draw("rag_pipeline.png")  # saves a diagram

# ── 6. Run the pipeline ──────────────────────────────────────────────────────
# result = rag_pipeline.run({
#     "text_embedder": {"text": "What is Haystack?"},
#     "bm25_retriever": {"query": "What is Haystack?"},
#     "prompt_builder": {"question": "What is Haystack?"},
#     "ranker": {"query": "What is Haystack?"},
# })
# print(result["llm"]["replies"][0])

print("Haystack 2.0 hybrid RAG pipeline constructed.")
print("Components:", list(rag_pipeline.graph.nodes))

TransformersSimilarityRanker is considered legacy and will no longer receive updates. It may be deprecated in a future release, with removal following after a deprecation period. Consider using SentenceTransformersSimilarityRanker instead, which provides the same functionality along with additional features.


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Haystack 2.0 hybrid RAG pipeline constructed.
Components: ['text_embedder', 'embedding_retriever', 'bm25_retriever', 'joiner', 'ranker', 'prompt_builder', 'llm']


### Haystack: Pros & Cons

| Pros | Cons |
|------|------|
| Explicit graph-based pipeline great for debugging | More verbose than LangChain/LlamaIndex |
| Built-in evaluation framework | Smaller ecosystem of integrations |
| Production-battle-tested at deepset | Steeper initial learning curve |
| First-class hybrid search support | Haystack 1.x → 2.x migration is breaking |
| Strong Elasticsearch/OpenSearch integration | Less popular in tutorial content |

## 4. RAGFlow

RAGFlow (by InfiniFlow) takes a fundamentally different approach: instead of treating documents as raw text, it performs **deep document understanding** before chunking.

### Core differentiator: Layout-aware parsing

Most RAG systems convert PDFs to text and lose all layout information. RAGFlow preserves:
- **Table structure** cells, headers, multi-column tables extracted as structured data
- **Figure captions** images paired with their descriptions
- **Document hierarchy** section headers, subsections, numbered lists
- **Reading order** correct left-to-right, top-to-bottom sequence in multi-column PDFs

### Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                         RAGFlow                             │
│                                                             │
│  Document Upload                                            │
│       │                                                     │
│       ▼                                                     │
│  Deep Document Parser (OCR + Layout Recognition)            │
│       │                                                     │
│       ▼                                                     │
│  Chunking Strategies:                                       │
│    - Naive  - Q&A  - Resume  - Manual  - Paper  - Book      │
│       │                                                     │
│       ▼                                                     │
│  Hybrid Retrieval (dense + sparse + re-rank)                │
│       │                                                     │
│       ▼                                                     │
│  Answer Generation with cited sources                       │
└─────────────────────────────────────────────────────────────┘
```

### Deployment

RAGFlow is self-hosted via Docker Compose. It includes a web UI for document management, a REST API for integration, and supports multiple LLM backends (OpenAI, Ollama, local models).

```bash
# Quick start
git clone https://github.com/infiniflow/ragflow.git
cd ragflow/docker
docker compose up -d
# Access at http://localhost
```

### When to choose RAGFlow

- Your documents are complex PDFs (research papers, financial reports, manuals)
- Table extraction is critical for your use case
- You want a complete, self-hosted solution with a UI
- You need cited answers with source highlighting
- You want production-ready RAG without writing pipeline code

## 5. Dify

Dify is an **LLMOps platform** it goes beyond RAG to provide a complete environment for building, deploying, and monitoring LLM applications.

### Architecture

```
┌──────────────────────────────────────────────────────────────────┐
│                            Dify Platform                         │
│                                                                  │
│   ┌──────────────┐  ┌───────────────┐  ┌─────────────────────┐  │
│   │  Workflow    │  │  Knowledge    │  │   Model Management  │  │
│   │  Builder     │  │  Base         │  │   (LLM/Embed/Rerank)│  │
│   │  (visual)    │  │  (vector DB)  │  │                     │  │
│   └──────┬───────┘  └───────┬───────┘  └──────────┬──────────┘  │
│          │                  │                     │              │
│          └──────────────────┼─────────────────────┘              │
│                             │                                    │
│   ┌─────────────────────────▼──────────────────────────────┐    │
│   │                    Workflow Engine                      │    │
│   │   LLM → Tool → Code → HTTP → Knowledge Retrieval → ... │    │
│   └─────────────────────────┬──────────────────────────────┘    │
│                             │                                    │
│   ┌─────────────────────────▼──────────────────────────────┐    │
│   │              API / Chatbot / Agent UI                   │    │
│   └────────────────────────────────────────────────────────┘    │
└──────────────────────────────────────────────────────────────────┘
```

### Key features

- **Visual workflow builder** drag-and-drop nodes: LLM, retrieval, code, HTTP request, condition branches, iteration
- **Knowledge base management** upload docs, configure chunking, select embedding model, monitor index status
- **Model management** swap LLMs per workflow node, supports 100+ models via unified interface
- **Prompt engineering IDE** version control for prompts, A/B testing
- **Observability** logs, traces, token usage, latency per run
- **API-first** every app you build gets a REST API automatically
- **Multi-tenant** workspaces, teams, role-based access control

### Deployment

```bash
# Self-hosted
git clone https://github.com/langgenius/dify.git
cd dify/docker
cp .env.example .env
docker compose up -d

# Or use Dify Cloud: https://cloud.dify.ai
```

### When to choose Dify

- Non-technical stakeholders need to iterate on prompts or workflows
- You want an integrated platform (no gluing together separate tools)
- You need observability and logging out of the box
- You're building internal tools or chatbots for business users

## 6. Visual RAG Builders: Flowise & Langflow

Both Flowise and Langflow provide drag-and-drop interfaces for building LLM pipelines, targeting users who want to prototype without writing code.

### Flowise

- Built on LangChain under the hood
- Node types map directly to LangChain components
- Export flows as JSON, import/share with team
- Deploy as API endpoint with one click
- Self-hosted (Node.js) or Flowise Cloud
- Marketplace of community flows

```bash
npx flowise start
# or: docker run -d -p 3000:3000 flowiseai/flowise
```

### Langflow

- Built on LangChain, React Flow UI
- More polished UI than Flowise
- Native Python components (write custom nodes in Python)
- DataStax (Astra DB) backing since acquisition
- Langflow Cloud available

```bash
pip install langflow
langflow run
```

### Visual builders vs code: When to use which

| Situation | Use Visual Builder | Use Code |
|-----------|-------------------|----------|
| Rapid prototype | ✅ | |
| Non-technical team member | ✅ | |
| Demo to stakeholders | ✅ | |
| Complex custom logic | | ✅ |
| Unit testing & CI/CD | | ✅ |
| Version control (Git diffs) | | ✅ |
| Performance tuning | | ✅ |
| Production at scale | | ✅ |
| Explain to non-devs how it works | ✅ | |

### Comparison

| Feature | Flowise | Langflow |
|---------|---------|----------|
| Backend | Node.js | Python |
| Custom nodes | Limited | Full Python support |
| UI polish | Good | Excellent |
| Community | Large | Large |
| Cloud option | Yes | Yes (DataStax) |
| License | Apache 2.0 | MIT |
| Best for | Quick LangChain prototypes | Python-heavy teams |

## 7. Other Notable Frameworks

### txtai

An all-in-one NLP platform built around embeddings. Unlike LangChain/LlamaIndex, txtai bundles embeddings, vector search, and NLP pipelines (summarization, translation, classification) into a single lightweight library.

```python
from txtai import Embeddings

# Create an embeddings index
embeddings = Embeddings({"path": "sentence-transformers/nli-mpnet-base-v2"})
embeddings.index([(0, "Python is a programming language", None),
                  (1, "RAG combines retrieval with generation", None)])
results = embeddings.search("What is RAG?", 1)
```

**Best for:** Lightweight local RAG, edge deployments, embedded NLP in Python apps.

---

### Semantic Kernel (Microsoft)

Enterprise-grade SDK for .NET and Python. Key differentiator: the **Planner** an LLM-powered orchestrator that assembles sequences of **Skills** (plugins) to accomplish a goal.

```python
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.core_plugins import TextMemoryPlugin

kernel = Kernel()
kernel.add_service(OpenAIChatCompletion(service_id="chat", ai_model_id="gpt-4o-mini"))
# Skills/plugins can be: Python functions, semantic (prompt-based), or OpenAPI specs
```

**Best for:** Enterprise .NET shops, complex multi-step agent orchestration, Azure integration.

---

### Vercel AI SDK

TypeScript SDK for building streaming AI interfaces in React/Next.js. Focuses on the **UI layer**: streaming text, tool calls rendered as React components, RSC (React Server Components) integration.

```typescript
import { streamText } from 'ai';
import { openai } from '@ai-sdk/openai';

const result = streamText({
  model: openai('gpt-4o-mini'),
  messages,
  tools: { /* tool definitions */ },
});
```

**Best for:** Next.js/React AI applications, streaming chat UIs, full-stack TypeScript teams.

---

### AnythingLLM

A desktop application for RAG over your documents. Features: multi-user, workspace isolation, document management UI, supports local (Ollama) and cloud LLMs. No coding required.

**Best for:** Teams wanting a self-hosted, UI-based RAG app without any code.

---

### PrivateGPT

Privacy-first local RAG. All processing stays on-device: local LLM (llama.cpp), local embeddings, local vector store. Zero cloud calls. REST API for integration.

```bash
git clone https://github.com/zylon-ai/private-gpt
poetry install --extras "ui llms-ollama embeddings-ollama vector-stores-qdrant"
PGPT_PROFILES=ollama make run
```

**Best for:** Air-gapped environments, legal/medical data, GDPR-strict organizations.

---

### LocalGPT

A simpler CLI-based local RAG. Ingest your documents, run queries all local with llama.cpp or GPT4All backends. Less configurable than PrivateGPT but easier to set up.

**Best for:** Personal knowledge base, quick local experiments.

---

### GPT4All

Cross-platform desktop app + Python library for running local LLMs. Has built-in LocalDocs RAG feature. Supports Windows/Mac/Linux. Community model repository.

```python
from gpt4all import GPT4All
model = GPT4All("Meta-Llama-3-8B-Instruct.Q4_0.gguf")
with model.chat_session():
    print(model.generate("What is RAG?", max_tokens=200))
```

**Best for:** Non-technical users wanting local LLMs, consumer hardware.

## 8. Document Preprocessing Libraries

Before any RAG framework can do its job, documents must be parsed into clean text. This is harder than it sounds PDFs especially are a disaster of binary format, scanned images, and layout tricks.

### Unstructured

The most comprehensive open-source document parsing library. One interface for 30+ file types.

- `partition()` auto-detects file type
- `partition_pdf()` PDF with optional OCR, layout detection
- `partition_docx()` Word documents with styles
- `partition_pptx()` PowerPoint slides
- `partition_html()` web pages
- `partition_image()` scanned images via OCR
- Returns typed elements: `Title`, `NarrativeText`, `Table`, `ListItem`, `Image`

### LlamaParse

Cloud API by LlamaIndex for advanced PDF parsing. Handles complex tables, multi-column layouts, equations (LaTeX), and embedded charts. Returns clean Markdown. Free tier available.

### Marker

Fast PDF → Markdown converter using ML models. Layout-aware, handles columns and tables. Can run locally on GPU. Faster than LlamaParse for batch processing.

```bash
pip install marker-pdf
marker_single input.pdf output/ --batch_multiplier 2
```

### Docling (IBM)

IBM Research's document understanding toolkit. Parses PDF, DOCX, PPTX, HTML, images into a unified DoclingDocument structure with layout, tables, and reading order. Integrates with LangChain and LlamaIndex.

```bash
pip install docling
```

In [4]:
# ── Document Preprocessing with Unstructured ─────────────────────────────────
# Install: pip install unstructured[pdf,docx,pptx] poppler-utils tesseract-ocr

# NOTE: Unstructured has system-level dependencies (poppler, tesseract).
# For production: use unstructured-ingest or the Unstructured Platform API.

# ── Basic usage ──────────────────────────────────────────────────────────────
# from unstructured.partition.auto import partition
# from unstructured.partition.pdf import partition_pdf
# from unstructured.partition.docx import partition_docx
# from unstructured.partition.html import partition_html
# from unstructured.staging.base import elements_to_json, elements_to_dicts
# from unstructured.cleaners.core import clean, clean_extra_whitespace
# from unstructured.documents.elements import (
#     Title, NarrativeText, Table, ListItem, Image, Header, Footer
# )

# ── PDF partitioning ─────────────────────────────────────────────────────────
# elements = partition_pdf(
#     filename="document.pdf",
#     strategy="hi_res",          # 'fast', 'ocr_only', 'hi_res'
#     # hi_res uses a layout detection model (detectron2)
#     infer_table_structure=True,  # extract tables as HTML
#     include_page_breaks=True,    # add PageBreak elements
#     languages=["eng"],           # OCR language hint
# )

# ── DOCX partitioning ────────────────────────────────────────────────────────
# elements = partition_docx(
#     filename="document.docx",
#     include_page_breaks=False,
# )

# ── Inspect elements ─────────────────────────────────────────────────────────
# for element in elements:
#     print(f"{type(element).__name__}: {str(element)[:80]}")
#     print(f"  metadata: {element.metadata}")

# ── Filter by element type ───────────────────────────────────────────────────
# titles = [e for e in elements if isinstance(e, Title)]
# tables = [e for e in elements if isinstance(e, Table)]
# narrative = [e for e in elements if isinstance(e, NarrativeText)]

# Tables come with HTML representation for structure preservation:
# for table in tables:
#     print(table.metadata.text_as_html)  # <table><tr><td>...</td></tr></table>

# ── Export to JSON for downstream processing ────────────────────────────────
# json_str = elements_to_json(elements)  # full serialization
# dicts = elements_to_dicts(elements)    # Python dicts

# ── Clean text ──────────────────────────────────────────────────────────────
# for element in elements:
#     element.apply(clean_extra_whitespace)  # removes extra whitespace in-place
#     element.apply(clean)                   # general cleaning

# ── Integration with LangChain ───────────────────────────────────────────────
# LangChain's UnstructuredFileLoader uses Unstructured under the hood:
# from langchain_community.document_loaders import UnstructuredFileLoader
# loader = UnstructuredFileLoader("document.pdf", mode="elements")
# docs = loader.load()  # each element becomes a Document

# ── Integration with LlamaIndex ──────────────────────────────────────────────
# from llama_index.readers.file import UnstructuredReader
# reader = UnstructuredReader()
# documents = reader.load_data(file=Path("document.pdf"))

# ── Demo: simulate element output ────────────────────────────────────────────
# Simulated output to show what Unstructured returns
simulated_elements = [
    {"type": "Title", "text": "Introduction to RAG", "metadata": {"page_number": 1}},
    {"type": "NarrativeText", "text": "RAG combines retrieval with generation...", "metadata": {"page_number": 1}},
    {"type": "Table", "text": "Framework | Stars | Language", "metadata": {"page_number": 2, "text_as_html": "<table>...</table>"}},
    {"type": "ListItem", "text": "LangChain: 90k+ GitHub stars", "metadata": {"page_number": 2}},
    {"type": "ListItem", "text": "LlamaIndex: 35k+ GitHub stars", "metadata": {"page_number": 2}},
    {"type": "Image", "text": "", "metadata": {"page_number": 3, "image_path": "/tmp/figure_1.png"}},
]

print("Simulated Unstructured output:")
for el in simulated_elements:
    print(f"  [{el['type']}] {el['text'][:60]}")

# ── LlamaParse example ───────────────────────────────────────────────────────
# pip install llama-parse
# from llama_parse import LlamaParse
# from llama_index.core import SimpleDirectoryReader
#
# parser = LlamaParse(result_type="markdown")  # or 'text'
# file_extractor = {".pdf": parser}
# documents = SimpleDirectoryReader(
#     input_files=["complex_report.pdf"],
#     file_extractor=file_extractor
# ).load_data()
# print(documents[0].text[:500])  # clean Markdown output

print("\nDocument preprocessing libraries demonstrated.")

Simulated Unstructured output:
  [Title] Introduction to RAG
  [NarrativeText] RAG combines retrieval with generation...
  [Table] Framework | Stars | Language
  [ListItem] LangChain: 90k+ GitHub stars
  [ListItem] LlamaIndex: 35k+ GitHub stars
  [Image] 

Document preprocessing libraries demonstrated.


## 9. Framework Comparison & Decision Matrix

### Full comparison table

| Framework | Open Source | Local/Private | Prod Ready | Learning Curve | Key Strength | Best For |
|-----------|-------------|---------------|------------|----------------|--------------|----------|
| **LangChain** | ✅ | ✅ | ⚠️ Medium | Medium | Ecosystem breadth | General LLM apps, prototyping |
| **LlamaIndex** | ✅ | ✅ | ✅ | Medium | Data indexing variety | Document Q&A, structured data |
| **Haystack** | ✅ | ✅ | ✅ High | High | Explicit pipelines | Enterprise NLP, hybrid search |
| **RAGFlow** | ✅ | ✅ | ✅ High | Low | Layout-aware parsing | Complex PDFs, tables, figures |
| **Dify** | ✅ | ✅ | ✅ High | Low | LLMOps platform | Teams, internal tools, no-code |
| **Flowise** | ✅ | ✅ | ⚠️ Medium | Very Low | Visual builder | Rapid prototyping, demos |
| **Langflow** | ✅ | ✅ | ⚠️ Medium | Very Low | Python + visual | Python teams, visual flows |
| **txtai** | ✅ | ✅ | ✅ | Low | Lightweight all-in-one | Edge, embedded, small scale |
| **Semantic Kernel** | ✅ | ✅ | ✅ High | High | Enterprise + .NET | .NET orgs, Azure, complex agents |
| **Vercel AI SDK** | ✅ | ❌ | ✅ High | Low (TS) | Streaming UI | Next.js/React AI apps |
| **AnythingLLM** | ✅ | ✅ | ⚠️ | Very Low | Desktop app | Non-technical users |
| **PrivateGPT** | ✅ | ✅✅ | ✅ | Low | Air-gapped | Privacy-strict orgs |
| **GPT4All** | ✅ | ✅✅ | ⚠️ | Very Low | Cross-platform desktop | Consumer, offline use |

### Decision flowchart

```
START: What do you need?
│
├─ No code / visual interface?
│   ├─ Need production + monitoring → Dify
│   ├─ Need desktop app → AnythingLLM
│   └─ Prototyping → Flowise or Langflow
│
├─ Must stay fully local / private?
│   ├─ Air-gapped / strict privacy → PrivateGPT
│   ├─ Simple CLI → LocalGPT
│   └─ Consumer desktop → GPT4All
│
├─ Complex PDF/document parsing needed?
│   └─ RAGFlow (or Unstructured + any framework)
│
├─ TypeScript / React / Next.js stack?
│   └─ Vercel AI SDK (+ LangChain.js or LlamaIndex.TS)
│
├─ .NET / enterprise Microsoft stack?
│   └─ Semantic Kernel
│
├─ Python, writing code:
│   ├─ Primary need: data indexing & querying → LlamaIndex
│   ├─ Primary need: production NLP pipeline → Haystack
│   ├─ Primary need: general LLM + agents + tools → LangChain
│   └─ Primary need: lightweight embedded NLP → txtai
│
└─ Don't know yet → LangChain (biggest community, most tutorials)
```

## 10. Side-by-Side Comparison: LangChain vs LlamaIndex

The same RAG task load a PDF, chunk it, embed it, store it, query it implemented in both frameworks. This highlights the API design philosophy differences.

**Task:** Build a Q&A system over a PDF document.

In [5]:
# ════════════════════════════════════════════════════════════════════════════
# SIDE-BY-SIDE: LangChain vs LlamaIndex
# Same task: PDF → chunk → embed → store → query
# ════════════════════════════════════════════════════════════════════════════

# ── LANGCHAIN APPROACH ──────────────────────────────────────────────────────
def build_rag_langchain(pdf_path: str, query: str) -> str:
    """
    LangChain philosophy:
    - Explicit pipeline construction with | operator (LCEL)
    - You wire together components: loader → splitter → embedder → store → retriever → chain
    - Full control over every step
    - Components are generic Runnables easy to swap any piece
    """
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_community.vectorstores import FAISS
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser

    # Step 1: Load
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()  # one Document per page

    # Step 2: Split
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, chunk_overlap=200
    )
    docs = splitter.split_documents(pages)

    # Step 3: Embed + Store
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(docs, embeddings)
    # ↑ FAISS is in-memory; no persistence by default
    # Save: vectorstore.save_local("faiss_index")
    # Load: FAISS.load_local("faiss_index", embeddings)

    # Step 4: Retriever
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    # Step 5: Chain
    prompt = ChatPromptTemplate.from_template(
        "Context: {context}\n\nQuestion: {question}\n\nAnswer:"
    )
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # LCEL chain reads left to right
    chain = (
        {"context": retriever | (lambda docs: "\n".join(d.page_content for d in docs)),
         "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return chain.invoke(query)
    # LangChain key insight: every step is a Runnable.
    # Swap FAISS → Chroma → Pinecone without changing anything else.
    # Swap ChatOpenAI → ChatAnthropic → ChatOllama same interface.


# ── LLAMAINDEX APPROACH ─────────────────────────────────────────────────────
def build_rag_llamaindex(pdf_path: str, query: str) -> str:
    """
    LlamaIndex philosophy:
    - Data-centric: documents → nodes → index → query engine
    - Higher-level abstractions hide pipeline details
    - Settings object configures globally (no passing llm/embeddings everywhere)
    - Index is a first-class citizen with its own query capabilities
    """
    from llama_index.core import VectorStoreIndex, Settings
    from llama_index.core import SimpleDirectoryReader
    from llama_index.readers.file import PDFReader
    from llama_index.llms.openai import OpenAI
    from llama_index.embeddings.openai import OpenAIEmbedding
    import os

    # Global config set once, applies everywhere
    Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
    Settings.embed_model = OpenAIEmbedding()
    Settings.chunk_size = 1000
    Settings.chunk_overlap = 200

    # Step 1: Load (LlamaIndex has its own reader ecosystem)
    reader = PDFReader()
    documents = reader.load_data(file=pdf_path)

    # Steps 2+3+4: Split + Embed + Store ALL handled by from_documents()
    # LlamaIndex is more opinionated: the index knows how to chunk itself
    index = VectorStoreIndex.from_documents(
        documents,
        show_progress=True,
    )
    # Persist: index.storage_context.persist("./storage")
    # Load:    index = load_index_from_storage(StorageContext.from_defaults(persist_dir="./storage"))

    # Step 5: Query engine (abstracts retriever + synthesizer together)
    query_engine = index.as_query_engine(
        similarity_top_k=4,
        response_mode="compact",  # fit context in as few LLM calls as possible
    )

    response = query_engine.query(query)
    return str(response)
    # LlamaIndex key insight: the Response object has .source_nodes
    # for provenance you know WHICH chunks contributed to the answer.


# ── KEY DIFFERENCES ─────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════╗
║           LangChain vs LlamaIndex: Key Differences           ║
╠══════════════════════════════╦═══════════════════════════════╣
║ LangChain                    ║ LlamaIndex                    ║
╠══════════════════════════════╬═══════════════════════════════╣
║ Explicit LCEL pipe chains    ║ Implicit pipeline in index    ║
║ You wire every connection    ║ from_documents() does it all  ║
║ llm/embeddings per object    ║ Settings object is global     ║
║ Returns str/AIMessage        ║ Returns Response with sources ║
║ More flexible / verbose      ║ More concise / opinionated    ║
║ Better for agent workflows   ║ Better for pure RAG/Q&A       ║
║ 100+ document loaders        ║ 50+ data connectors (Llama Hub)║
║ chain.stream() built-in      ║ query_engine.stream() support  ║
╚══════════════════════════════╩═══════════════════════════════╝
""")

# When to choose:
# LangChain: building agents, need fine control, general LLM app beyond RAG
# LlamaIndex: pure document Q&A, metadata filtering, multiple index types needed


╔══════════════════════════════════════════════════════════════╗
║           LangChain vs LlamaIndex: Key Differences           ║
╠══════════════════════════════╦═══════════════════════════════╣
║ LangChain                    ║ LlamaIndex                    ║
╠══════════════════════════════╬═══════════════════════════════╣
║ Explicit LCEL pipe chains    ║ Implicit pipeline in index    ║
║ You wire every connection    ║ from_documents() does it all  ║
║ llm/embeddings per object    ║ Settings object is global     ║
║ Returns str/AIMessage        ║ Returns Response with sources ║
║ More flexible / verbose      ║ More concise / opinionated    ║
║ Better for agent workflows   ║ Better for pure RAG/Q&A       ║
║ 100+ document loaders        ║ 50+ data connectors (Llama Hub)║
║ chain.stream() built-in      ║ query_engine.stream() support  ║
╚══════════════════════════════╩═══════════════════════════════╝



## Additional Learning Resources

### Official Documentation

| Framework | Docs | GitHub |
|-----------|------|--------|
| LangChain | https://python.langchain.com | github.com/langchain-ai/langchain |
| LlamaIndex | https://docs.llamaindex.ai | github.com/run-llama/llama_index |
| Haystack | https://docs.haystack.deepset.ai | github.com/deepset-ai/haystack |
| RAGFlow | https://ragflow.io/docs | github.com/infiniflow/ragflow |
| Dify | https://docs.dify.ai | github.com/langgenius/dify |
| Flowise | https://docs.flowiseai.com | github.com/FlowiseAI/Flowise |
| Langflow | https://docs.langflow.org | github.com/langflow-ai/langflow |
| Unstructured | https://unstructured-io.github.io | github.com/Unstructured-IO/unstructured |
| Semantic Kernel | https://learn.microsoft.com/semantic-kernel | github.com/microsoft/semantic-kernel |
| txtai | https://neuml.github.io/txtai | github.com/neuml/txtai |

### Key Papers

1. **RAG (2020)** Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"
   - https://arxiv.org/abs/2005.11401
   - The foundational paper that named and formalized RAG

2. **REALM (2020)** Guu et al., "REALM: Retrieval-Augmented Language Model Pre-Training"
   - https://arxiv.org/abs/2002.08909
   - Pre-training with retrieval, not just inference-time

3. **FLARE (2023)** Jiang et al., "Active Retrieval Augmented Generation"
   - https://arxiv.org/abs/2305.06983
   - Retrieve only when the model is uncertain

4. **Self-RAG (2023)** Asai et al., "Self-RAG: Learning to Retrieve, Generate, and Critique"
   - https://arxiv.org/abs/2310.11511
   - Model learns when and what to retrieve

5. **RAPTOR (2024)** Sarthi et al., "RAPTOR: Recursive Abstractive Processing for Tree-Organized Retrieval"
   - https://arxiv.org/abs/2401.18059
   - Tree-based index with recursive summarization

### Courses & Tutorials

- **DeepLearning.AI**: "LangChain for LLM Application Development", "Building and Evaluating Advanced RAG"
- **LlamaIndex**: Official blog and cookbook at https://docs.llamaindex.ai/en/stable/examples/
- **Haystack**: Tutorial series at https://haystack.deepset.ai/tutorials
- **Full Stack LLM Bootcamp** (UC Berkeley): https://fullstackdeeplearning.com/llm-bootcamp/

### Community

- LangChain Discord: ~100k members
- LlamaIndex Discord: ~25k members
- Haystack Discord: ~8k members
- r/LangChain, r/LocalLLaMA (Reddit)
- Awesome-LLM (GitHub): curated list of LLM resources
- LLM Papers of the Week newsletter

In [6]:
# ── Summary: Framework Selection Quick Reference ─────────────────────────────

framework_guide = {
    "I'm just starting with RAG": [
        "LangChain largest community, most tutorials",
        "LlamaIndex cleaner API if your focus is document Q&A",
    ],
    "I need to parse complex PDFs (tables, figures)": [
        "RAGFlow layout-aware parsing built-in",
        "Unstructured best Python library for document parsing",
        "LlamaParse cloud API, excellent for research papers",
        "Docling (IBM) open-source, structured document output",
    ],
    "I need a no-code / visual solution": [
        "Dify full LLMOps platform, production-ready",
        "RAGFlow self-hosted, includes web UI",
        "Flowise / Langflow drag-and-drop pipeline builders",
        "AnythingLLM desktop app, zero code",
    ],
    "Privacy is critical, must stay fully local": [
        "PrivateGPT zero cloud, REST API",
        "LocalGPT simple CLI",
        "GPT4All cross-platform desktop",
        "Ollama + LangChain/LlamaIndex local LLMs, full framework",
    ],
    "Building for production at scale": [
        "Haystack explicit pipelines, built-in evaluation",
        "LangChain + LangSmith (observability)",
        "Dify monitoring, logging, A/B testing built-in",
    ],
    "TypeScript / React / Next.js": [
        "Vercel AI SDK streaming, RSC, hooks",
        "LangChain.js TypeScript port",
        "LlamaIndex.TS TypeScript port",
    ],
    ".NET / C# / Enterprise Microsoft": [
        "Semantic Kernel first-class .NET support, Azure integration",
    ],
}

print("=" * 60)
print("  RAG FRAMEWORK SELECTION GUIDE")
print("=" * 60)
for situation, recommendations in framework_guide.items():
    print(f"\n{situation}:")
    for rec in recommendations:
        print(f"  → {rec}")
print("\n" + "=" * 60)
print("Remember: the best framework is the one your team will actually use.")
print("Start simple, measure, then optimize.")

  RAG FRAMEWORK SELECTION GUIDE

I'm just starting with RAG:
  → LangChain largest community, most tutorials
  → LlamaIndex cleaner API if your focus is document Q&A

I need to parse complex PDFs (tables, figures):
  → RAGFlow layout-aware parsing built-in
  → Unstructured best Python library for document parsing
  → LlamaParse cloud API, excellent for research papers
  → Docling (IBM) open-source, structured document output

I need a no-code / visual solution:
  → Dify full LLMOps platform, production-ready
  → RAGFlow self-hosted, includes web UI
  → Flowise / Langflow drag-and-drop pipeline builders
  → AnythingLLM desktop app, zero code

Privacy is critical, must stay fully local:
  → PrivateGPT zero cloud, REST API
  → LocalGPT simple CLI
  → GPT4All cross-platform desktop
  → Ollama + LangChain/LlamaIndex local LLMs, full framework

Building for production at scale:
  → Haystack explicit pipelines, built-in evaluation
  → LangChain + LangSmith (observability)
  → Dify monitoring,